# 13 — Validación automática de riesgos

Este notebook ejecuta un segundo LLM como **validador independiente**. Evalúa los 156 candidatos contra el texto fuente, compara sus decisiones con las 55 etiquetas humanas disponibles y genera una muestra dirigida para control humano.

Las etiquetas humanas no se incorporan al prompt: se reservan para evaluación.

In [ ]:
!pip -q install pandas pyarrow openpyxl openai
!rm -rf /content/proyecto_ActividadGrado-Riesgos /content/resultados_validador
!git clone -q --branch dia-01-diagnostico-falsos-positivos https://github.com/nalpata/proyecto_ActividadGrado-Riesgos.git /content/proyecto_ActividadGrado-Riesgos
print('Repositorio preparado.')

## 1. Configurar la clave privada
La clave permanece únicamente en la sesión actual de Colab.

In [ ]:
import os
from getpass import getpass
api_key = getpass('Ingrese OPENAI_API_KEY: ')
if not api_key.strip(): raise ValueError('La clave no puede estar vacía.')
os.environ['OPENAI_API_KEY'] = api_key.strip()
del api_key
print('Clave cargada de forma temporal.')

## 2. Ejecutar el agente validador
Los candidatos se agrupan por chunk, por lo que se realizan aproximadamente 56 llamadas y no 156.

In [ ]:
import subprocess
from pathlib import Path
REPO = Path('/content/proyecto_ActividadGrado-Riesgos')
INPUT = REPO / 'data/evaluation/risk_validation/day_01/revision_asistida_riesgos_dia_01.xlsx'
CHUNKS = REPO / 'data/processed/chunks/chunks_recursive.parquet'
OUTPUT = Path('/content/resultados_validador')
command = ['python', str(REPO/'src/risk/validate_risks.py'), '--input-xlsx', str(INPUT), '--chunks', str(CHUNKS), '--output-dir', str(OUTPUT), '--sample-size', '35']
subprocess.run(command, check=True, env=os.environ.copy())

## 3. Revisar métricas preliminares
Estas métricas se calculan únicamente sobre los 55 registros etiquetados.

In [ ]:
import json
print('METADATOS')
print((OUTPUT/'validator_metadata.json').read_text(encoding='utf-8'))
print('\nMÉTRICAS SOBRE ETIQUETAS HUMANAS')
print((OUTPUT/'validator_metrics.json').read_text(encoding='utf-8'))

## 4. Descargar resultados
La muestra de control incluye desacuerdos, rechazos automáticos y casos de baja confianza. Adjunta el ZIP completo en el chat.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive('/content/resultados_validador_riesgos', 'zip', OUTPUT)
files.download(zip_path)
print('Descarga preparada:', zip_path)